## Introduccion a Lark

Lark es una libreria de parsing para Python que soporta gramaticas EBNF.
El proyecto utiliza Lark por las siguientes ventajas:

### Caracteristicas Principales

| Caracteristica | Descripcion |
|----------------|-------------|
| Parser LALR | Parsing eficiente O(n) |
| Propagacion de posicion | Linea y columna en cada token |
| Transformers | Conversion automatica a AST |
| Soporte Unicode | Identificadores en espanol |

### Archivo de Gramatica

La gramatica se encuentra en:

```
app/core/parser/grammar/pseudocode.lark
```

## Estructura de la Gramatica

La gramatica esta organizada en secciones logicas:

### 1. Programa Principal

```lark
?start: program

program: class_definition* algorithm
```

El simbolo `?` indica que el nodo debe ser transparente (no aparece en el parse tree).

### 2. Definicion de Algoritmo

```lark
algorithm: ALGORITHM_KW IDENTIFIER "(" parameter_list? ")" block

parameter_list: parameter ("," parameter)*

parameter: array_parameter
         | object_parameter  
         | IDENTIFIER
```

### 3. Bloque de Codigo

```lark
block: BEGIN statement_list END

statement_list: (statement)*

statement: variable_declaration
         | assignment
         | for_loop
         | while_loop
         | repeat_loop
         | if_statement
         | call_statement
         | return_statement
```

## Reglas Principales

### Asignacion

```lark
assignment: lvalue ASSIGN expression

lvalue: IDENTIFIER                           // variable simple
      | IDENTIFIER "[" expression "]"        // array access
      | IDENTIFIER "[" expression "]" "[" expression "]"  // 2D array
      | IDENTIFIER "." IDENTIFIER            // object field
```

El operador de asignacion (`ASSIGN`) acepta:
- Flecha izquierda: `<-`
- Dos puntos igual: `:=`

Se evita usar `=` para asignacion para no crear ambiguedad con comparacion.

### Ciclo For

```lark
for_loop: FOR IDENTIFIER ASSIGN expression TO expression DO block
```

Ejemplo de codigo que coincide:

```
for i <- 1 to n do
    // cuerpo
end
```

### Ciclo While

```lark
while_loop: WHILE "(" condition ")" DO block
```

Ejemplo:

```
while (i < n) do
    // cuerpo
end
```

### Ciclo Repeat

```lark
repeat_loop: REPEAT statement_list UNTIL "(" condition ")"
```

Ejemplo:

```
repeat
    // cuerpo
until (condicion)
```

### Condicional If

```lark
if_statement: IF "(" condition ")" THEN block (ELSE block)?
```

El bloque `else` es opcional (indicado por `?`).

## Terminales y Tokens

### Palabras Clave

Las palabras clave tienen prioridad `.2` para preceder sobre identificadores:

```lark
ALGORITHM_KW.2: "algorithm"i | "algoritmo"i | "proceso"i | "subproceso"i | "funcion"i
BEGIN.2: "begin"i | "inicio"i
END.2: "end"i | "fin"i | "finproceso"i | "finalgoritmo"i | "finfuncion"i
FOR.2: "for"i | "para"i
TO.2: "to"i | "hasta"i
DO.2: "do"i | "hacer"i
WHILE.2: "while"i | "mientras"i
REPEAT.2: "repeat"i | "repetir"i
UNTIL.2: "until"i | "hasta que"i
IF.2: "if"i | "si"i
THEN.2: "then"i | "entonces"i
ELSE.2: "else"i | "sino"i
CALL.2: "call"i | "llamar"i
RETURN.2: "return"i | "retornar"i | "devolver"i
```

El sufijo `i` hace que el match sea case-insensitive.

### Operadores Logicos

```lark
AND.2: "and"i | "&&"
OR.2: "or"i | "||"
NOT.2: "not"i | "no"i | "!"
```

Nota: Se evitan `y` y `o` como operadores porque causan conflictos con identificadores de una letra.

### Operadores de Comparacion

```lark
LT: "<"
GT: ">"
LE: "<=" | "<="
GE: ">=" | ">="
EQ: "="
NE: "!=" | "!="
```

Se soportan simbolos Unicode para comparacion (ej: `<=` para menor o igual).

### Funciones Especiales

```lark
CEIL.2: "ceil" | "ceil"i | "techo"i
FLOOR.2: "floor" | "floor"i | "piso"i
LENGTH.2: "length"i | "longitud"i | "tamano"i
```

## Prioridades y Ambiguedad

### Precedencia de Operadores

La gramatica define precedencia mediante la estructura de las reglas:

```lark
?expression: logical_or

?logical_or: logical_and
           | logical_or OR logical_and

?logical_and: logical_not
            | logical_and AND logical_not

?logical_not: comparison
            | NOT logical_not

?comparison: arithmetic
           | arithmetic compare_op arithmetic

?arithmetic: term
           | arithmetic ADD term
           | arithmetic SUB term

?term: factor
     | term MUL factor
     | term DIVIDE factor
     | term MOD factor

?factor: power
       | ADD factor
       | SUB factor

?power: atom
      | atom POW factor
```

### Orden de Precedencia (menor a mayor)

| Nivel | Operadores |
|-------|------------|
| 1 | OR |
| 2 | AND |
| 3 | NOT |
| 4 | <, >, <=, >=, =, != |
| 5 | +, - (aritmeticos) |
| 6 | *, /, % |
| 7 | +, - (unarios) |
| 8 | ^ (potencia) |

### Resolucion de Conflictos

Las prioridades numericas (`.1`, `.2`) resuelven conflictos:

```lark
// Palabras clave tienen prioridad .2
FOR.2: "for"i | "para"i

// Identificadores tienen prioridad .1 (menor)
IDENTIFIER.1: /[a-zA-Z_][a-zA-Z0-9_]*/
```

Esto asegura que `for` sea reconocido como palabra clave, no como identificador.

## Extensiones y Modificaciones

### Agregar Nueva Estructura de Control

Para agregar un ciclo `foreach`, se modificaria la gramatica:

```lark
// En la seccion de statements
statement: ...
         | foreach_loop

// Nueva regla
foreach_loop: FOREACH IDENTIFIER IN expression DO block

// Nuevo terminal
FOREACH.2: "foreach"i | "paracada"i
IN.2: "in"i | "en"i
```

### Agregar Nuevo Tipo de Dato

Para soportar diccionarios:

```lark
// En la seccion de atoms
?atom: ...
     | dictionary_access

dictionary_access: IDENTIFIER "{" expression "}"
```

### Agregar Operador

Para agregar el operador de division entera `//`:

```lark
// En la seccion de term
?term: ...
     | term INTDIV factor

// Nuevo terminal
INTDIV: "//"
```

## Gramatica Completa: Estructura de Expresiones

### Atomos (Nivel mas bajo)

```lark
?atom: NUMBER
     | BOOLEAN
     | STRING
     | IDENTIFIER
     | IDENTIFIER "[" expression "]"                    // array access
     | IDENTIFIER "[" expression "]" "[" expression "]" // 2D array
     | IDENTIFIER "." IDENTIFIER                        // object field
     | function_call
     | "(" expression ")"
     | CEIL "(" expression ")"                          // techo
     | FLOOR "(" expression ")"                         // piso
     | LENGTH "(" IDENTIFIER ")"                        // longitud
```

### Literales

```lark
BOOLEAN.2: /\b(TRUE|VERDADERO|FALSE|FALSO)\b/i

NUMBER: /\d+(\.\d+)?/
      | /\d+e[+-]?\d+/i       // notacion cientifica

STRING: /"[^"]*"/
      | /'[^']*'/
```

### Identificadores

```lark
IDENTIFIER.1: /[a-zA-Z_aeiouAEIOU][a-zA-Z0-9_aeiouAEIOU]*/

CLASS_NAME.1: /[A-Z][a-zA-Z0-9_]*/
```

Se soportan caracteres con acentos y ene para identificadores en espanol.

## Comentarios y Espacios

La gramatica ignora automaticamente:

```lark
COMMENT: /->>[^\n]*/        // Comentario estilo PSeInt
       | //[^\n]*/          // Comentario estilo C

NEWLINE: /[\r\n]+/

%ignore /[ \t]+/            // Espacios y tabs
%ignore COMMENT             // Comentarios
%ignore NEWLINE             // Saltos de linea
```

Esto permite que el codigo tenga cualquier formato de indentacion y comentarios
sin afectar el parsing.

## Demostracion: Parsing con Diferentes Sintaxis

In [ ]:
import sys
sys.path.insert(0, '../..')

from app.core.parser.pseudocode_parser import PseudocodeParser

parser = PseudocodeParser()

# Ejemplo en ingles
codigo_ingles = '''
algorithm sum(n)
begin
    total <- 0
    for i <- 1 to n do
        total <- total + i
    end
    return total
end
'''

# Ejemplo en espanol
codigo_espanol = '''
algoritmo suma(n)
inicio
    total <- 0
    para i <- 1 hasta n hacer
        total <- total + i
    fin
    retornar total
fin
'''

# Ambos deben parsear correctamente
ast_ingles = parser.parse(codigo_ingles)
ast_espanol = parser.parse(codigo_espanol)

print(f"Algoritmo en ingles: {ast_ingles.algorithm.name}")
print(f"Algoritmo en espanol: {ast_espanol.algorithm.name}")

---

## Conclusiones

La gramatica Lark proporciona:

1. **Flexibilidad bilingue**: Sintaxis en ingles y espanol
2. **Claridad**: Estructura EBNF legible y mantenible
3. **Eficiencia**: Parser LALR O(n)
4. **Extensibilidad**: Facil de agregar nuevas construcciones
5. **Robustez**: Manejo de prioridades y ambiguedades

---

**Siguiente notebook recomendado**: `02_complexity_analysis/big_o_examples.ipynb` para ver como se analiza la complejidad del AST.